In [17]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/xingtianma/econ3916/refs/heads/main/data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

In [18]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [19]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula,data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Thu, 26 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        18:58:24   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [20]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [21]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df["Home_Value"], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [22]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import plotly.express as px

# ==========================================
# 1. MOCK DATA & MODEL SETUP (Context)
# ==========================================
# Simulating a dataset for a hedonic pricing model (e.g., House Price = f(SqFt, Age))
np.random.seed(42)
n_samples = 500
sqft = np.random.uniform(800, 4000, n_samples)
age = np.random.uniform(1, 100, n_samples)
# Adding some heteroscedastic noise (variance increases with sqft)
true_price = 50000 + (150 * sqft) - (500 * age) + np.random.normal(0, 10 * sqft)

df = pd.DataFrame({'Price': true_price, 'SqFt': sqft, 'Age': age})

# Fit the OLS model using statsmodels
model = smf.ols('Price ~ SqFt + Age', data=df)
results = model.fit()

# ==========================================
# 2. RESIDUAL EXTRACTION & DASHBOARD LOGIC
# ==========================================

# MECHANISM CHECK: 
# We extract the predicted values (fittedvalues) and the error terms (resid) 
# directly from the statsmodels OLSResults object. These are stored as Pandas Series.
predicted_values = results.fittedvalues
residuals = results.resid

# MECHANISM CHECK:
# To find outliers exceeding 2 standard deviations, we first calculate the standard 
# deviation of the extracted residuals array. We then create a boolean mask (True/False) 
# by checking if the absolute value of each residual is greater than 2 * std_dev.
std_dev = residuals.std()
outlier_mask = np.abs(residuals) > (2 * std_dev)

# Combine into a DataFrame for Plotly to consume cleanly
plot_df = pd.DataFrame({
    'Predicted_Price': predicted_values,
    'Residual_Error': residuals,
    'Is_Outlier': outlier_mask
})

# Map the boolean mask to readable labels for the legend
plot_df['Status'] = plot_df['Is_Outlier'].map({True: '> 2 SD Outlier', False: 'Normal'})

# Create the interactive scatter plot using Plotly Express
fig = px.scatter(
    plot_df,
    x='Predicted_Price',
    y='Residual_Error',
    color='Status',
    # MECHANISM CHECK: We use color_discrete_map to strictly enforce the visual logic.
    # Outliers get the stark 'crimson' color, while normal residuals get a muted blue.
    color_discrete_map={'> 2 SD Outlier': 'crimson', 'Normal': '#1f77b4'},
    title='Hedonic Pricing OLS: Residual Forensics Dashboard',
    labels={'Predicted_Price': 'Predicted Values (Fitted)', 'Residual_Error': 'Residuals'},
    opacity=0.7,
    hover_data=['Predicted_Price', 'Residual_Error']
)

# Add the horizontal zero-line to represent perfect predictions
fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Zero Error Line")

# Display the interactive dashboard
fig.show()